In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, StrMethodFormatter
from IPython.display import display

project_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "data" / "processed" / "btc_spy_daily.csv").exists())
RESULTS_DIR = project_root / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TOTAL_BUDGET = 12_000.0
SELECTED_START_YEARS = [2014, 2016, 2018, 2020, 2022, 2024, 2025]
MIN_HOLDING_YEARS = 1
assert TOTAL_BUDGET > 0


In [ ]:
prices = pd.read_csv(project_root / "data" / "processed" / "btc_spy_daily.csv",
    index_col="date", parse_dates=True).rename(columns={"price_btc":"BTC", "price_spy":"SPY"})[["BTC","SPY"]]
assert prices.index.is_monotonic_increasing and prices.index.is_unique
assert np.isfinite(prices.to_numpy()).all() and prices.gt(0).all().all()
assert MIN_HOLDING_YEARS >= 1
end_date = prices.index[-1]
monthly_dates = prices.groupby(prices.index.to_period("M")).head(1).index
eligible_starts = pd.DatetimeIndex([d for d in monthly_dates
    if d + pd.DateOffset(years=MIN_HOLDING_YEARS) <= end_date])
assert len(eligible_starts) > 0
print(f"Common endpoint: {end_date.date()}; {len(eligible_starts)} eligible monthly starts")


In [ ]:
def simulate_deployment(price, purchase_dates, budget):
    """Compare buy-and-hold with equal installments; idle cash earns zero."""
    purchase_dates = pd.DatetimeIndex(purchase_dates)
    if len(purchase_dates) == 0 or not purchase_dates.is_unique:
        raise ValueError("Purchase dates must be nonempty and unique.")
    if not purchase_dates.isin(price.index).all():
        raise ValueError("Every purchase date must have an observed price.")
    deposits = pd.Series(0.0, index=price.index)
    deposits.loc[purchase_dates] = budget / len(purchase_dates)
    # Correct only floating-point summation residue at the final purchase.
    deposits.loc[purchase_dates[-1]] += budget - deposits.sum()
    deployed = deposits.cumsum()
    units = (deposits / price).cumsum()
    cash = budget - deployed
    invested = units * price
    history = pd.DataFrame({
        "Lump sum wealth": budget * price / price.iloc[0],
        "Monthly invested value": invested,
        "Monthly cash": cash,
        "Monthly total wealth": invested + cash,
        "Cumulative contributions": deployed,
        "Contribution": deposits,
    })
    return history


In [ ]:
def contribution_xirr(contributions, terminal_value, terminal_date):
    contributions = contributions[contributions > 0]
    elapsed = np.asarray((pd.Timestamp(terminal_date) - contributions.index).days / 365.25)
    amounts = contributions.to_numpy(dtype=float)
    if len(amounts) == 0 or np.any(elapsed < 0) or not np.any(elapsed > 0):
        raise ValueError("XIRR requires contributions before the terminal date.")
    if terminal_value <= amounts[elapsed == 0].sum():
        raise ValueError("No finite XIRR above -100% for these cash flows.")
    def residual(log_growth):
        terms = np.log(amounts) + elapsed * log_growth
        peak = terms.max()
        return peak + np.log(np.exp(terms - peak).sum()) - np.log(terminal_value)
    low, high = -1.0, 1.0
    for _ in range(60):
        if residual(low) <= 0 <= residual(high):
            break
        low *= 2
        high *= 2
    else:
        raise ValueError("Could not bracket the XIRR root.")
    for _ in range(200):
        mid = (low + high) / 2
        if residual(mid) > 0:
            high = mid
        else:
            low = mid
    return np.expm1((low + high) / 2)


In [ ]:
rows = []
for start in eligible_starts:
    segment = prices.loc[start:end_date]
    purchase_dates = monthly_dates[monthly_dates >= start]
    years = (end_date - start).days / 365.25
    for asset in prices:
        history = simulate_deployment(segment[asset], purchase_dates, TOTAL_BUDGET)
        for strategy, wealth_col in [("Lump sum", "Lump sum wealth"),
                                    ("Monthly", "Monthly total wealth")]:
            wealth = history[wealth_col]
            cashflows = history["Contribution"] if strategy == "Monthly" else pd.Series(
                [TOTAL_BUDGET], index=segment.index[:1])
            final_invested = history["Monthly invested value"].iloc[-1] if strategy == "Monthly" else wealth.iloc[-1]
            rows.append({"Start": start, "End": end_date, "Asset": asset, "Strategy": strategy,
                "Years": years, "Budget": TOTAL_BUDGET,
                "Purchases": len(purchase_dates) if strategy == "Monthly" else 1,
                "Payment": TOTAL_BUDGET / len(purchase_dates) if strategy == "Monthly" else TOTAL_BUDGET,
                "Final wealth": wealth.iloc[-1], "Profit": wealth.iloc[-1] - TOTAL_BUDGET,
                "Total return": wealth.iloc[-1] / TOTAL_BUDGET - 1,
                "Whole-account CAGR": (wealth.iloc[-1] / TOTAL_BUDGET) ** (1 / years) - 1,
                "Invested-sleeve XIRR": contribution_xirr(cashflows, final_invested, end_date),
                "Max drawdown": (wealth / wealth.cummax() - 1).min()})
all_results = pd.DataFrame(rows)
wealth_comparison = all_results.pivot(index=["Start", "Asset"], columns="Strategy", values="Final wealth")
wealth_comparison["Monthly relative advantage"] = wealth_comparison["Monthly"] / wealth_comparison["Lump sum"] - 1
print(f"Calculated {len(all_results):,} asset/strategy scenarios.")


In [ ]:
selected_dates = [eligible_starts[0]]
for year in SELECTED_START_YEARS:
    candidates = eligible_starts[eligible_starts >= pd.Timestamp(year=year, month=1, day=1)]
    if len(candidates) and candidates[0].year == year:
        selected_dates.append(candidates[0])
    else:
        print(f"Skipping {year}: no eligible start with sufficient remaining history.")
selected_dates = pd.DatetimeIndex(sorted(set(selected_dates)))
selected_results = all_results[all_results["Start"].isin(selected_dates)].copy()
selected_table = selected_results.set_index(["Start", "Asset", "Strategy"])[[
    "Years", "Purchases", "Payment", "Final wealth", "Whole-account CAGR", "Invested-sleeve XIRR", "Max drawdown"]]
formats = {"Years":"{:.2f}", "Payment":"${:,.2f}", "Final wealth":"${:,.2f}",
    "Whole-account CAGR":"{:.2%}", "Invested-sleeve XIRR":"{:.2%}", "Max drawdown":"{:.2%}"}
display(selected_table.style.format(formats))

# Direct BTC/SPY comparison for matching strategy and start.
asset_comparison = all_results.pivot(index=["Start", "Strategy"], columns="Asset", values="Final wealth")
asset_comparison["BTC / SPY final wealth"] = asset_comparison["BTC"] / asset_comparison["SPY"]
display(asset_comparison.loc[asset_comparison.index.get_level_values("Start").isin(selected_dates)]
        .style.format({"BTC":"${:,.2f}", "SPY":"${:,.2f}", "BTC / SPY final wealth":"{:.2f}"}))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for column, asset in enumerate(prices.columns):
    data = all_results[all_results["Asset"] == asset]
    for strategy in ["Lump sum", "Monthly"]:
        part = data[data["Strategy"] == strategy]
        axes[0,column].plot(part["Start"], part["Final wealth"], label=strategy)
        axes[1,column].plot(part["Start"], part["Whole-account CAGR"], label=strategy)
    axes[0,column].set_yscale("log")
    axes[0,column].set_ylabel("Final wealth (USD, log scale)")
    axes[1,column].set_ylabel("Whole-account CAGR")
    axes[1,column].yaxis.set_major_formatter(PercentFormatter(1))
    axes[1,column].axhline(0, color="black", linewidth=0.8)
    for ax in axes[:,column]:
        ax.set_title(asset)
        ax.set_xlabel("Scenario starting date")
        ax.legend()
        ax.grid(alpha=0.25)
fig.suptitle(f"Equal ${TOTAL_BUDGET:,.0f} budgets; common endpoint {end_date:%Y-%m-%d}")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "start_date_wealth_cagr.png", dpi=300)
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
for asset in prices:
    data = wealth_comparison.xs(asset, level="Asset")
    ax.plot(data.index, data["Monthly relative advantage"], label=asset)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Monthly deployment advantage by starting date (above zero favors monthly)")
ax.set_xlabel("Scenario starting date")
ax.set_ylabel("Monthly / lump-sum final wealth − 1")
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "start_date_strategy_advantage.png", dpi=300)
plt.show()


In [ ]:
cutoff_rows = []
for cutoff in selected_dates:
    for asset in prices:
        subset = all_results[(all_results["Start"] >= cutoff) & (all_results["Asset"] == asset)]
        comp = wealth_comparison.xs(asset, level="Asset").loc[cutoff:]
        gap = comp["Monthly relative advantage"]
        ties = np.isclose(gap, 0, atol=1e-10, rtol=0)
        cutoff_rows.append({"Cutoff": cutoff, "Asset": asset, "Starts": len(comp),
            "First start": comp.index.min(), "Last start": comp.index.max(),
            "Monthly win rate": ((gap > 0) & ~ties).mean(),
            "Lump win rate": ((gap < 0) & ~ties).mean(), "Tie rate": ties.mean(),
            "Median monthly advantage": gap.median(),
            "Median lump CAGR": subset.loc[subset["Strategy"] == "Lump sum", "Whole-account CAGR"].median(),
            "Median monthly CAGR": subset.loc[subset["Strategy"] == "Monthly", "Whole-account CAGR"].median()})
cutoff_summary = pd.DataFrame(cutoff_rows).set_index(["Cutoff","Asset"])
cutoff_formats = {col:"{:.2%}" for col in cutoff_summary if "rate" in col or "Median" in col}
cutoff_formats.update({"First start":"{:%Y-%m-%d}","Last start":"{:%Y-%m-%d}"})
display(cutoff_summary.style.format(cutoff_formats))


In [ ]:
all_results.to_csv(RESULTS_DIR / "start_date_all_scenarios.csv", index=False)
selected_table.to_csv(RESULTS_DIR / "start_date_selected_summary.csv")
wealth_comparison.to_csv(RESULTS_DIR / "start_date_strategy_comparison.csv")
asset_comparison.to_csv(RESULTS_DIR / "start_date_asset_comparison.csv")
cutoff_summary.to_csv(RESULTS_DIR / "start_date_cutoff_summary.csv")
assumptions = {"budget":TOTAL_BUDGET, "common_end":str(end_date.date()),
    "minimum_calendar_years":MIN_HOLDING_YEARS, "selected_start_years":SELECTED_START_YEARS,
    "idle_cash_yield":0, "costs":0, "fractional_units":True,
    "schedule":"equal installments on first available shared date each month; budget fixed, payment varies",
    "comparison":"common endpoint; holding and deployment durations vary with starting date",
    "xirr":"invested sleeve; excludes idle cash", "spy":"adjusted-price dividend-reinvestment proxy"}
(RESULTS_DIR / "start_date_assumptions.json").write_text(json.dumps(assumptions,indent=2))
report = ('<!doctype html><html><head><meta charset="utf-8"><title>Start-date sensitivity</title>'
    '<style>body{font-family:system-ui;max-width:1400px;margin:35px auto;padding:20px;}'
    'table{border-collapse:collapse;font-size:12px;}td,th{padding:7px;border-bottom:1px solid #ddd;}'
    'img{max-width:100%;}</style></head><body><h1>BTC vs SPY: excluding early starting dates</h1>'
    f'<p>Each scenario starts with ${TOTAL_BUDGET:,.0f}; all end {end_date:%Y-%m-%d}. '
    f'At least {MIN_HOLDING_YEARS} calendar year(s) of history required.</p>'
    '<p>Idle cash earns 0%; no costs or taxes; fractional units allowed. All money is available upfront. '
    'Later starts imply fewer, larger installments. CAGR includes idle cash; invested-sleeve XIRR does not.</p>'
    '<h2>Selected starts</h2>' + selected_table.style.format(formats).to_html()
    + '<img src="start_date_wealth_cagr.png" alt="Wealth and CAGR across starting dates">'
    '<img src="start_date_strategy_advantage.png" alt="Monthly versus lump-sum advantage">'
    '<h2>BTC versus SPY for selected starts</h2>'
    + asset_comparison.loc[asset_comparison.index.get_level_values("Start").isin(selected_dates)].style.format(
        {"BTC":"${:,.2f}","SPY":"${:,.2f}","BTC / SPY final wealth":"{:.2f}"}).to_html()
    + '<h2>Exclude earlier starts</h2>' + cutoff_summary.style.format(cutoff_formats).to_html()
    + '<p>Start-date and holding-duration effects are intertwined. These nested, overlapping scenarios are '
    'descriptive, not independent probabilities or forecasts. Notebook 09 supplies complementary fixed-horizon comparisons. '
    'SPY adjusted prices proxy reinvested dividends; shared closes may miss intraday/weekend extremes.</p></body></html>')
(RESULTS_DIR / "start_date_report.html").write_text(report,encoding="utf-8")
print(f"Saved five CSVs, assumptions JSON, two charts and an HTML report to {RESULTS_DIR}")
